# Oppitunti 11 - **Model Context Protocol (MCP)**

**Model Context Protocol (MCP)** on avoin standardi, joka mahdollistaa agenttien dynaamisen työkalujen, resurssien ja tietolähteiden löytämisen ja käytön suoritusaikana. Sen sijaan, että työkalut kovakoodattaisiin agenttiin, MCP antaa agenttien yhdistää ulkoisiin palvelimiin, jotka tarjoavat ominaisuuksia tarpeen mukaan.

Tässä oppitunnissa opit:
- Mikä MCP on ja miksi se on tärkeä agenttijärjestelmille
- Miten MCP:n asiakas-palvelinarkkitehtuuri toimii
- Miten rakentaa agenteja, jotka käyttävät MCP-tyylistä työkalujen löytämistä


## Asennus

**Esivaatimukset:**
- Azure AI Foundry -projekti, jossa on otettu käyttöön malli
- Suorita `az login` todennusta varten


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mikä on Model Context Protocol (MCP)?

MCP määrittelee standardoidun tavan, jolla tekoälyagentit löytävät ja vuorovaikuttavat ulkoisten työkalujen ja datalähteiden kanssa:

- **MCP Server**: Tarjoaa työkaluja, resursseja ja kehotteita standardoidun protokollan kautta
- **MCP Client**: Agentin ajonaikainen ympäristö, joka yhdistää palvelimiin ja löytää käytettävissä olevat ominaisuudet
- **Dynaaminen löytäminen**: Agentit eivät tarvitse kovakoodattuja työkaluja — ne löytävät käytettävissä olevat ominaisuudet ajonaikana

Tämä on tehokas tapa rakentaa laajennettavia agenttijärjestelmiä, joissa uusia ominaisuuksia voidaan lisätä muuttamatta agenttikoodia.


## Kuinka MCP toimii

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agentti (MCP-asiakas) muodostaa yhteyden MCP-palvelimeen
2. Palvelin vastaa listalla saatavilla olevista työkaluista ja niiden skeemoista
3. Agentti voi sitten kutsua mitä tahansa löydettyä työkalua päättelynsä aikana
4. Tulokset palautuvat saman protokollan kautta


## MCP-työkalujen löytymisen simulointi

Koska oikea MCP-palvelin vaatii käynnissä olevan palvelinprosessin, näytämme mallin käyttämällä `@tool`-funktioita, jotka simuloivat sitä, mitä MCP:hen liitetty majoituspalvelu tarjoaisi.

Tuotannossa nämä työkalut löydettäisiin dynaamisesti MCP-palvelimelta sen sijaan, että ne määriteltäisiin paikallisesti.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Agentin rakentaminen MCP-tyylisillä työkaluilla


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP tuotannossa

Tuotantoympäristössä MCP mahdollistaa tehokkaita käytäntöjä:

- **Dynaaminen työkalujen löytäminen**: Agentit muodostavat yhteyden MCP-palvelimiin ja löytävät työkaluja ajonaikaisesti
- **Irrotettu arkkitehtuuri**: Työkalujen tarjoajat voivat päivittää itsenäisesti riippumatta agentista
- **Organisaatioiden välinen jakaminen**: Tiimit voivat altistaa ominaisuuksia MCP-palvelimien kautta, joita mikä tahansa agentti voi käyttää
- **Microsoft Agent Framework -tuki**: MAF sisältää sisäänrakennetun MCP-asiakastuen `mcp`-integraation kautta

Käyttääksesi oikeaa MCP-palvelinta MAF:n kanssa, muodostat yhteyden `hosted_mcp_tool()`-funktion tai MCP-asiakasintegraation kautta.

**Lisätietoja:**
- [MCP-määritys](https://modelcontextprotocol.io/)
- [Microsoft Agent Frameworkin MCP-tuki](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Yhteenveto

Tässä oppitunnissa opit:
- **MCP** on avoin standardi, joka mahdollistaa agenttien ja työkalutoimittajien välisen dynaamisen työkalujen löytämisen
- **Asiakas–palvelinarkkitehtuuri** antaa agenteille mahdollisuuden löytää ominaisuuksia ajonaikaisesti
- MCP mahdollistaa **laajennettavat, löyhästi kytketyt agenttijärjestelmät**, joihin työkaluja voidaan lisätä ilman koodimuutoksia
- Microsoft Agent Framework tarjoaa tuotantokäyttöön **sisäänrakennetun MCP-tuen**


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vastuuvapauslauseke:
Tämä asiakirja on käännetty käyttäen tekoälykäännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme täsmällisyyteen, huomioithan, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäisellä kielellä tulee pitää virallisena lähteenä. Kriittisten tietojen osalta suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkintavirheistä.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
